This document analyzes the dataset logged by CARLA's system. The metrics to be measured are the following: 
1. **Reaction Time**
    - This is from the time the alert was sent until the time the driver made an action. This will help the researches know the appropriate time to give an alert. 
2. **Completion Time**
    - This measures the duration of the whole violation. From the alert being triggered until the speeding gets resolved. 
3. **Ignored Alert** 
    - This checks whether the driver made an action to the alert or ignored it. This will be used to prove the effectiveness of providing an alert to decrease speed.
4. **Speed Change**
    - This checks the speed. 
5. **Alert Effectiveness**
    - This computes for the number of resolved speeding violations, where the rider either *reduced throttle* or *brakes* which is then divided by all violations. (resolved/all violations) 

*Note: An action made by the driver can be either a release of the throttle or pressing on the brakes.* 

In [109]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [139]:
df = pd.read_csv("cleaned_Sim.csv")

df.head()


,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_start,violation_end,violation_state,event_start,event_id
0,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,0,0,0
1,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,0,0,0
2,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,0,0,0
3,1.771924e+09,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,...",30.0,False,False,0,0,0
4,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,False,False,0,0,0


In [140]:
df.columns

Index(['timestamp', 'phase', 'scenario', 'speed_kmh', 'event', 'details',
       'speed_limit', 'violation_start', 'violation_end', 'violation_state',
       'event_start', 'event_id'],
      dtype='object')

In [142]:
df["scenario"].unique()

array(['WEBSOCKET', 'TRAFFIC_LIGHT', 'SPEED_LIMIT', 'STOP', 'COLLISION',
       'HUD'], dtype=object)

In [143]:
df.loc[
    (df["event_id"]==5),
    ["speed_kmh", "speed_limit", "details", "event_id"]
]

,speed_kmh,speed_limit,details,event_id
13107,37.29,30.0,"{""phase"": 3, ""speed"": 37.29, ""speed_limit"": 30...",5
13108,37.51,30.0,"{""phase"": 3, ""speed"": 37.51, ""speed_limit"": 30...",5
13109,37.88,30.0,"{""phase"": 3, ""speed"": 37.88, ""speed_limit"": 30...",5
13110,37.88,30.0,"{""phase"": 3, ""speed"": 37.88, ""speed_limit"": 30...",5
13111,38.18,30.0,"{""phase"": 3, ""speed"": 38.18, ""speed_limit"": 30...",5
...,...,...,...,...
13176,37.75,30.0,"{""phase"": 3, ""speed"": 37.75, ""speed_limit"": 30...",5
13177,37.75,30.0,"{""phase"": 3, ""speed"": 37.75, ""speed_limit"": 30...",5
13178,37.75,30.0,"{""phase"": 3, ""speed"": 37.75, ""speed_limit"": 30...",5
13179,48.73,30.0,Speed 48.73 km/h > limit 30.00 km/h at Locatio...,5


In [144]:
df.loc[
    df["event_id"].isin([183]),
    ["phase", "event_start", "speed_kmh", "speed_limit", "details", "event_id"]
]

,phase,event_start,speed_kmh,speed_limit,details,event_id
187498,2,1,97.61,90.0,"{""phase"": 2, ""speed"": 97.61, ""speed_limit"": 90...",183
187499,2,0,98.60,90.0,"{""phase"": 2, ""speed"": 98.6, ""speed_limit"": 90,...",183
187500,2,0,98.60,90.0,"{""phase"": 2, ""speed"": 98.6, ""speed_limit"": 90,...",183
187501,2,0,98.60,90.0,"{""phase"": 2, ""speed"": 98.6, ""speed_limit"": 90,...",183
187502,2,0,99.06,90.0,"{""phase"": 2, ""speed"": 99.06, ""speed_limit"": 90...",183
187503,2,0,100.72,90.0,"{""phase"": 2, ""speed"": 100.72, ""speed_limit"": 9...",183
187504,2,0,101.29,90.0,"{""phase"": 2, ""speed"": 101.29, ""speed_limit"": 9...",183
187505,2,0,101.29,90.0,"{""phase"": 2, ""speed"": 101.29, ""speed_limit"": 9...",183
187506,2,0,101.29,90.0,"{""phase"": 2, ""speed"": 101.29, ""speed_limit"": 9...",183
187507,2,0,102.71,90.0,"{""phase"": 2, ""speed"": 102.71, ""speed_limit"": 9...",183


In [145]:
df.loc[df["speed_limit"] == 60, "event_id"].unique()

array([  4,   0,  17,  38, 111, 144, 164, 183, 184, 185, 186, 220, 282,
       292, 372])

## Metrics Computation & Analysis

In [146]:
df_Metrics = df.copy()
df_Metrics

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_start,violation_end,violation_state,event_start,event_id
0,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,0,0,0
1,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,0,0,0
2,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,False,False,0,0,0
3,1.771924e+09,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,...",30.0,False,False,0,0,0
4,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,False,False,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
446324,1.771682e+09,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,...",30.0,False,False,0,0,0
446325,1.771682e+09,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,...",30.0,False,False,0,0,0
446326,1.771682e+09,3,WEBSOCKET,3.41,WS_SEND,"{""phase"": 3, ""speed"": 3.41, ""speed_limit"": 30,...",30.0,False,False,0,0,0
446327,1.771682e+09,3,STOP,674.87,PHASE_STOP,Assisted-H,NaN,False,False,0,0,0


### Reaction Time

In [147]:
# look for details == "action=REDUCE_THROTTLE"
df_Metrics["slow_down"] = df_Metrics["details"].str.contains("REDUCE_THROTTLE", na=False)

*action=REDUCE_THROTTLE* is a signal that the driver released yung tapak niya sa pedal. 

In [148]:
# get timestamp and event_id
event_start_time = df_Metrics[df_Metrics["event_start"] == 1].groupby("event_id")["timestamp"].first()

reaction_time = df_Metrics[df_Metrics["slow_down"]].groupby("event_id")["timestamp"].first()

In [149]:
reaction_timeK = reaction_time - event_start_time

In [188]:
display(reaction_timeK.head(2))

event_id
0     NaN
1    0.73
Name: timestamp, dtype: float64

In [182]:
df.loc[
    df["event_id"].isin([19]),
    ["event_start", "speed_kmh", "speed_limit", "details", "event_id"]
]

,event_start,speed_kmh,speed_limit,details,event_id
73761,1,39.36,30.0,"{""phase"": 2, ""speed"": 39.36, ""speed_limit"": 30...",19
73762,0,40.60,30.0,"{""phase"": 2, ""speed"": 40.6, ""speed_limit"": 30,...",19
73763,0,41.21,30.0,"{""phase"": 2, ""speed"": 41.21, ""speed_limit"": 30...",19
73764,0,41.21,30.0,"{""phase"": 2, ""speed"": 41.21, ""speed_limit"": 30...",19
73765,0,41.21,30.0,"{""phase"": 2, ""speed"": 41.21, ""speed_limit"": 30...",19
73766,0,43.06,30.0,"{""phase"": 2, ""speed"": 43.06, ""speed_limit"": 30...",19
73767,0,46.38,30.0,"{""phase"": 2, ""speed"": 46.38, ""speed_limit"": 30...",19
73768,0,47.58,30.0,"{""phase"": 2, ""speed"": 47.58, ""speed_limit"": 30...",19
73769,0,47.58,30.0,"{""phase"": 2, ""speed"": 47.58, ""speed_limit"": 30...",19
73770,0,47.58,30.0,"{""phase"": 2, ""speed"": 47.58, ""speed_limit"": 30...",19


In [183]:
df_Metrics.loc[
    df_Metrics["details"].str.contains("REDUCE_THROTTLE", na=False),
    "reaction_time"
] = df_Metrics["details"].str.extract(r"time=(\d+\.\d+)s").astype(float)

In [189]:
df_Metrics["carla_reaction_time"] = df_Metrics["details"].str.extract(
    r"action=REDUCE_THROTTLE\|time=(\d+\.\d+)s"
).astype(float)

In [193]:
carla_reactions = df_Metrics.loc[
    df_Metrics["carla_reaction_time"].notna(),
    ["event_id", "carla_reaction_time"]
]

In [195]:
display(carla_reactions.tail(20))

,event_id,carla_reaction_time
429131,374,0.54
429255,375,1.30
429689,376,0.80
430179,377,0.86
430258,0,0.27
430349,378,0.18
430632,379,0.59
430695,0,0.03
432905,380,0.36
437861,0,0.24


In [186]:
df_Metrics["reaction_time"].head(20)

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
14   NaN
15   NaN
16   NaN
17   NaN
18   NaN
19   NaN
Name: reaction_time, dtype: float64

*Note: in the cases of NaN values: at times the rider is just above the speeding threshold, the system in CARLA might not have detected enough throttle release. Therefore, in the subtracting the action to the start becomes not applicable.*

In [184]:
reaction_table = reaction_time.reset_index(name="reaction_time")

### Completion Time

In [155]:
# get timestamp and event_id
event_end_time = (
    df_Metrics[df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False)]
    .groupby("event_id")["timestamp"]
    .first()
)

In [156]:
completion_time = event_end_time - event_start_time

In [175]:
display(completion_time.head(20))

event_id
0      NaN
1     1.30
2     2.26
3     1.29
4     0.23
5     4.04
6     0.86
7     1.58
8     0.92
9     2.12
10    4.44
11    5.31
12    1.21
13    1.30
14    3.84
15    1.30
16    0.89
17    0.22
18    0.51
19    1.54
Name: timestamp, dtype: float64

### Ignored Alert

In [158]:
ignored = (reaction_timeK >= 2.5).astype(int)

In [159]:
reaction_table["ignored"] = (
    (reaction_table["reaction_time"] >= 2.5)
).astype(int)

In [160]:
display(reaction_table.tail(15))

,event_id,reaction_time,ignored
376,376,0.75,0
377,377,0.83,0
378,378,0.16,0
379,379,0.56,0
380,380,0.34,0
381,381,NaN,0
382,382,0.33,0
383,383,2.50,1
384,384,NaN,0
385,385,0.76,0


### Speed Change

### Alert Effectiveness

In [161]:
speedingCount = df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False).sum()
total_events = df_Metrics["event_id"].max()

In [162]:
print("Speeding Count: ", speedingCount)
print("Total events: ", total_events)

Speeding Count:  611
Total events:  390


In [163]:
effective = speedingCount/total_events

In [164]:
print(effective)

1.5666666666666667


## Alert-Phase Analysis

In [135]:
# Transfer to Sim Analysis

#Baseline
df_NC = df[df['phase'] == 1]
display(df_NC)

# Audio Only 
df_AO = df[df['phase'] == 2]
#display(df_AO)

# Haptic Only
df_HO = df[df['phase'] == 3]
#display(df_HO)

# Combination Haptic and Audio
df_HA = df[df['phase'] == 4]
#display(df_HA)


,timestamp,phase,speed_kmh,event,details,speed_limit,violation_start,violation_end,violation_state,event_start,event_id
7122,1.771924e+09,1,37.14,REACTION,action=REDUCE_THROTTLE|time=0.21s|control={'th...,NaN,False,False,0,0,0
7123,1.771924e+09,1,37.14,SPEED_VIOLATION,Speed 37.14 km/h > limit 30.00 km/h at Locatio...,30.0,False,False,0,0,0
7124,1.771924e+09,1,36.53,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,NaN,False,True,0,0,0
7125,1.771924e+09,1,14.79,YELLOW_LIGHT_PASS,"Passed yellow at Location(x=-85.075447, y=113....",NaN,False,False,0,0,0
7126,1.771924e+09,1,19.89,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=-22.61465...,NaN,False,False,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
437869,1.771680e+09,1,36.00,REACTION,action=VIOLATION_RESOLVED|time=1.78s|reaction_...,NaN,False,True,0,0,0
437870,1.771680e+09,1,37.01,SPEED_VIOLATION,Speed 37.01 km/h > limit 30.00 km/h at Locatio...,30.0,False,False,0,0,0
437871,1.771680e+09,1,36.70,REACTION,action=VIOLATION_RESOLVED|time=0.15s|reaction_...,NaN,False,True,0,0,0
437872,1.771680e+09,1,475.51,PHASE_STOP,Baseline,NaN,False,False,0,0,0


In [136]:
#speeding_count = df["overspeed"].sum()
#VIOLATION COUNTS
speeding_count = df[df["event"] == "SPEED_VIOLATION"].shape[0]
stoplight_countRed = df[df["event"] == "RED_LIGHT_VIOLATION"].shape[0]
stoplight_countYellow = df[df["event"] == "YELLOW_LIGHT_PASS"].shape[0]
print("Speed count: ", speeding_count)
print("Red Light count: ", stoplight_countRed)
print("Yellow Light count: ", stoplight_countYellow)

Speed count:  926
Red Light count:  44033
Yellow Light count:  91


In [167]:
resolved_rows = df_Metrics[df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False)]
resolved_rows.head(10)

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_start,violation_end,violation_state,event_start,event_id,slow_down,alert_sent
780,1.771924e+09,4,SPEED_LIMIT,36.30,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,NaN,False,True,1,0,1,False,False
5506,1.771924e+09,4,SPEED_LIMIT,36.39,REACTION,action=VIOLATION_RESOLVED|time=2.30s|reaction_...,NaN,False,True,1,0,2,False,False
6265,1.771924e+09,4,SPEED_LIMIT,35.52,REACTION,action=VIOLATION_RESOLVED|time=1.32s|reaction_...,NaN,False,True,1,0,3,False,False
7124,1.771924e+09,1,SPEED_LIMIT,36.53,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,NaN,False,True,0,0,0,False,False
7131,1.771925e+09,1,SPEED_LIMIT,36.93,REACTION,action=VIOLATION_RESOLVED|time=5.16s|reaction_...,NaN,False,True,0,0,0,False,False
11708,1.771925e+09,3,SPEED_LIMIT,66.34,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,NaN,False,True,1,0,4,False,False
13180,1.771925e+09,3,SPEED_LIMIT,35.19,REACTION,action=VIOLATION_RESOLVED|time=4.07s|reaction_...,NaN,False,True,1,0,5,False,False
15600,1.771926e+09,2,SPEED_LIMIT,36.85,REACTION,action=VIOLATION_RESOLVED|time=2.60s|reaction_...,NaN,False,True,0,0,0,False,False
15603,1.771926e+09,2,SPEED_LIMIT,96.73,REACTION,action=VIOLATION_RESOLVED|time=0.45s|reaction_...,NaN,False,True,0,0,0,False,False
15609,1.771926e+09,2,SPEED_LIMIT,36.87,REACTION,action=VIOLATION_RESOLVED|time=0.99s|reaction_...,NaN,False,True,0,0,0,False,False


In [169]:
df.loc[
    df["details"].str.contains("VIOLATION_RESOLVED", na=False),
    ["scenario", "speed_kmh", "speed_limit", "event", "details", "event_id"]
].head(20)

,scenario,speed_kmh,speed_limit,event,details,event_id
780,SPEED_LIMIT,36.30,NaN,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,1
5506,SPEED_LIMIT,36.39,NaN,REACTION,action=VIOLATION_RESOLVED|time=2.30s|reaction_...,2
6265,SPEED_LIMIT,35.52,NaN,REACTION,action=VIOLATION_RESOLVED|time=1.32s|reaction_...,3
7124,SPEED_LIMIT,36.53,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,0
7131,SPEED_LIMIT,36.93,NaN,REACTION,action=VIOLATION_RESOLVED|time=5.16s|reaction_...,0
11708,SPEED_LIMIT,66.34,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,4
13180,SPEED_LIMIT,35.19,NaN,REACTION,action=VIOLATION_RESOLVED|time=4.07s|reaction_...,5
15600,SPEED_LIMIT,36.85,NaN,REACTION,action=VIOLATION_RESOLVED|time=2.60s|reaction_...,0
15603,SPEED_LIMIT,96.73,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.45s|reaction_...,0
15609,SPEED_LIMIT,36.87,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.99s|reaction_...,0
